<a href="https://colab.research.google.com/github/Miranita-ar/Skripsi-Gojek-App-Review/blob/main/Code/(XB)_Skripsi_IndoBERT_(Purposive_Sampling_XAI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXPLAINABLE ARTIFICIAL INTELLIGENCE (XAI)

## Analisis Sentimen Ulasan Aplikasi Gojek
### Menggunakan Model IndoBERT Base P2

---

**Peneliti** : Miranita Anisa Rohmah

**Universitas** : UIN Syarif Hidayatullah Jakarta

---

### Tahap Penelitian

Notebook **XB**

**Purposive Sampling untuk Explainable Artificial Intelligence (XAI)**

---

Versi : **1.0**

Update terakhir :

*(Jika ada revisi menjadi 1.1, 1.2, dst.)*

---

Notebook ini bertujuan memilih sampel ulasan dari data uji menggunakan teknik **purposive sampling** berdasarkan hasil prediksi model IndoBERT. Sampel yang dipilih akan digunakan pada Notebook XC (SHAP) dan Notebook XD (LIME).

1. Informasi Project

2. Persiapan
   2.1 Install Library
   2.2 Import Library
   2.3 Mount Google Drive
   2.4 Konfigurasi Fungsi
   2.5 Konfigurasi Path
   2.6 Konfigurasi Folder XB

3. Load Data
   3.1 Membaca test_predictions_xai.csv
   3.2 Validasi Dataset

4. Analisis Distribusi
   4.1 Distribusi Prediction Case
   4.2 Distribusi Confidence

5. Purposive Sampling
   5.1 Menentukan Target Sampling
   5.2 Sampling Otomatis
   5.3 Validasi Hasil Sampling

6. Simpan Hasil

7. Ringkasan Notebook

Notebook XB

 - BAB 1  Informasi Project
 - BAB 2  Persiapan
 - BAB 3  Load Data
 - BAB 4  Analisis Distribusi
 - BAB 5  Purposive Sampling   ← Bagian terpenting
 - BAB 6  Validasi Sampel
 - BAB 7  Simpan Hasil
 - BAB 8  Ringkasan Notebook

# 2. Persiapan

## 2.1 Install Library

In [ ]:
# =====================================================
# CELL 1 : INSTALL LIBRARY
# =====================================================

!pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q sentencepiece

## 2.2 Import Library

In [ ]:
# =====================================================
# CELL 2 : IMPORT LIBRARY
# =====================================================

import os
import random
import warnings

import numpy as np
import pandas as pd

from google.colab import drive

warnings.filterwarnings("ignore")

## 2.3 Mount Google Drive

In [ ]:
# =====================================================
# CELL 3 : MOUNT GOOGLE DRIVE
# =====================================================

drive.mount("/content/drive")

print("=" * 60)
print("GOOGLE DRIVE BERHASIL DI-MOUNT")
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GOOGLE DRIVE BERHASIL DI-MOUNT


## 2.4 Konfigurasi Fungsi

In [ ]:
# =====================================================
# CELL 4 : FUNGSI BANTUAN
# =====================================================

def print_header(title):
    print("=" * 60)
    print(title)
    print("=" * 60)


def print_success(text):
    print(f"✅ {text}")


def print_info(text):
    print(f"ℹ️ {text}")


def print_warning(text):
    print(f"⚠️ {text}")

## 2.5 Konfigurasi Path

In [ ]:
# =====================================================
# CELL 5 : PATH PROJECT
# =====================================================

PROJECT_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT"

print_header("PROJECT DIRECTORY")

print_info(PROJECT_DIR)

PROJECT DIRECTORY
ℹ️ /content/drive/MyDrive/Skripsi_IndoBERT


## 2.6 Konfigurasi Folder XB

In [ ]:
# =====================================================
# CELL 6 : KONFIGURASI FOLDER
# =====================================================

print_header("KONFIGURASI FOLDER XB")

XAI_DIR = os.path.join(
    PROJECT_DIR,
    "xai"
)

XB_DIR = os.path.join(
    XAI_DIR,
    "XB_Purposive_Sampling"
)

TABLE_DIR = os.path.join(
    XB_DIR,
    "tables"
)

FIGURE_DIR = os.path.join(
    XB_DIR,
    "figures"
)

LOG_DIR = os.path.join(
    XB_DIR,
    "logs"
)

OUTPUT_DIR = os.path.join(
    XB_DIR,
    "output"
)

for folder in [

    XAI_DIR,

    XB_DIR,

    TABLE_DIR,

    FIGURE_DIR,

    LOG_DIR,

    OUTPUT_DIR

]:

    os.makedirs(
        folder,
        exist_ok=True
    )

print_success("Folder Notebook XB berhasil dibuat")

print()

print_info(f"XB_DIR      : {XB_DIR}")
print_info(f"TABLE_DIR   : {TABLE_DIR}")
print_info(f"FIGURE_DIR  : {FIGURE_DIR}")
print_info(f"LOG_DIR     : {LOG_DIR}")
print_info(f"OUTPUT_DIR  : {OUTPUT_DIR}")

KONFIGURASI FOLDER XB
✅ Folder Notebook XB berhasil dibuat

ℹ️ XB_DIR      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Purposive_Sampling
ℹ️ TABLE_DIR   : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Purposive_Sampling/tables
ℹ️ FIGURE_DIR  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Purposive_Sampling/figures
ℹ️ LOG_DIR     : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Purposive_Sampling/logs
ℹ️ OUTPUT_DIR  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Purposive_Sampling/output


In [ ]:
# =====================================================
# CEK LOKASI FILE test_predictions_xai.csv
# =====================================================

import os

print("Mencari file test_predictions_xai.csv ...\n")

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        if "test_predictions" in file.lower():

            print(os.path.join(root, file))

Mencari file test_predictions_xai.csv ...

/content/drive/MyDrive/Skripsi_IndoBERT/xai/XA_Persiapan_XAI/test_predictions_xai.csv


# 3. Memuat Hasil Prediksi Notebook XA

## 3.1 Lokasi File Hasil Prediksi

In [ ]:
# =====================================================
# CELL 7 : LOKASI FILE HASIL PREDIKSI
# =====================================================

print_header("LOKASI FILE HASIL PREDIKSI")

PREDICTION_FILE = os.path.join(
    PROJECT_DIR,
    "xai",
    "XA_Persiapan_XAI",
    "test_predictions_xai.csv"
)

print_info(f"Path File : {PREDICTION_FILE}")

print()

if os.path.exists(PREDICTION_FILE):

    print_success("File hasil prediksi ditemukan.")

else:

    raise FileNotFoundError(
        f"File tidak ditemukan:\n{PREDICTION_FILE}"
    )

LOKASI FILE HASIL PREDIKSI
ℹ️ Path File : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XA_Persiapan_XAI/test_predictions_xai.csv

✅ File hasil prediksi ditemukan.


## 3.2 Membaca Hasil Prediksi

In [ ]:
# =====================================================
# CELL 8 : MEMBACA HASIL PREDIKSI
# =====================================================

print_header("MEMBACA HASIL PREDIKSI")

prediction_df = pd.read_csv(
    PREDICTION_FILE
)

print_success("Data berhasil dimuat")

print()

print_info(f"Jumlah Data : {len(prediction_df):,}")

print_info(f"Jumlah Kolom : {prediction_df.shape[1]}")

MEMBACA HASIL PREDIKSI
✅ Data berhasil dimuat

ℹ️ Jumlah Data : 1,980
ℹ️ Jumlah Kolom : 12


## 3.3 Validasi Dataset

In [ ]:
# =====================================================
# CELL 9 : VALIDASI DATASET
# =====================================================

print_header("VALIDASI DATASET")

print("Nama Kolom")

print("-" * 40)

for column in prediction_df.columns:
    print(column)

print()

print("5 Data Pertama")

display(
    prediction_df.head()
)

print_success("Dataset valid")

VALIDASI DATASET
Nama Kolom
----------------------------------------
text
actual_label
actual_sentiment
predicted_label
predicted_sentiment
prob_negatif
prob_netral
prob_positif
confidence
correct
prediction_case
confidence_level

5 Data Pertama


,text,actual_label,actual_sentiment,predicted_label,predicted_sentiment,prob_negatif,prob_netral,prob_positif,confidence,correct,prediction_case,confidence_level
0,pelayanan oke buat drivernya maaf baru pertama...,2,Positif,2,Positif,0.001382,0.002627,0.995991,0.995991,True,Benar_Positif,Tinggi
1,3 kali dikecewakan driver,0,Negatif,0,Negatif,0.963739,0.025840,0.010421,0.963739,True,Benar_Negatif,Tinggi
2,top banget deh pokoknya aku saranin kalian jug...,2,Positif,2,Positif,0.001092,0.000922,0.997986,0.997986,True,Benar_Positif,Tinggi
3,membantu sekali buat perjalanan jauh dekat,2,Positif,2,Positif,0.000912,0.000955,0.998133,0.998133,True,Benar_Positif,Tinggi
4,sukses selalu bwt pra driver,2,Positif,2,Positif,0.001337,0.000952,0.997711,0.997711,True,Benar_Positif,Tinggi


✅ Dataset valid


## 3.4 Informasi Dataset

In [ ]:
# =====================================================
# CELL 10 : INFORMASI DATASET
# =====================================================

print_header("INFORMASI DATASET")

display(
    prediction_df.info()
)

print()

display(
    prediction_df.describe(
        include="all"
    )
)

INFORMASI DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   text                 1980 non-null   object 
 1   actual_label         1980 non-null   int64  
 2   actual_sentiment     1980 non-null   object 
 3   predicted_label      1980 non-null   int64  
 4   predicted_sentiment  1980 non-null   object 
 5   prob_negatif         1980 non-null   float64
 6   prob_netral          1980 non-null   float64
 7   prob_positif         1980 non-null   float64
 8   confidence           1980 non-null   float64
 9   correct              1980 non-null   bool   
 10  prediction_case      1980 non-null   object 
 11  confidence_level     1980 non-null   object 
dtypes: bool(1), float64(4), int64(2), object(5)
memory usage: 172.2+ KB


None

,text,actual_label,actual_sentiment,predicted_label,predicted_sentiment,prob_negatif,prob_netral,prob_positif,confidence,correct,prediction_case,confidence_level
count,1980,1980.000000,1980,1980.000000,1980,1980.000000,1980.000000,1980.000000,1980.000000,1980,1980,1980
unique,1496,NaN,3,NaN,3,NaN,NaN,NaN,NaN,2,9,3
top,mantap,NaN,Positif,NaN,Positif,NaN,NaN,NaN,NaN,True,Benar_Positif,Tinggi
freq,86,NaN,1292,NaN,1274,NaN,NaN,NaN,NaN,1790,1228,1681
mean,NaN,1.342929,NaN,1.310101,NaN,0.305546,0.041540,0.652914,0.941805,NaN,NaN,NaN
std,NaN,0.919210,NaN,0.938643,NaN,0.422589,0.103307,0.450889,0.125934,NaN,NaN,NaN
min,NaN,0.000000,NaN,0.000000,NaN,0.000760,0.000821,0.003182,0.362289,NaN,NaN,NaN
25%,NaN,0.000000,NaN,0.000000,NaN,0.001020,0.000911,0.048267,0.963620,NaN,NaN,NaN
50%,NaN,2.000000,NaN,2.000000,NaN,0.001247,0.000958,0.997813,0.997813,NaN,NaN,NaN
75%,NaN,2.000000,NaN,2.000000,NaN,0.861837,0.022577,0.998066,0.998066,NaN,NaN,NaN


## 3.5 Missing Value

In [ ]:
# =====================================================
# CELL 11 : PEMERIKSAAN MISSING VALUE
# =====================================================

print_header("PEMERIKSAAN MISSING VALUE")

missing = prediction_df.isnull().sum()

display(
    missing.to_frame(
        "Jumlah Missing"
    )
)

print()

if missing.sum() == 0:

    print_success(
        "Tidak ditemukan missing value."
    )

else:

    print_warning(
        "Masih terdapat missing value."
    )

PEMERIKSAAN MISSING VALUE


,Jumlah Missing
text,0
actual_label,0
actual_sentiment,0
predicted_label,0
predicted_sentiment,0
prob_negatif,0
prob_netral,0
prob_positif,0
confidence,0
correct,0



✅ Tidak ditemukan missing value.


## 3.6 Validasi Prediction Case

In [ ]:
# =====================================================
# CELL 12 : VALIDASI PREDICTION CASE
# =====================================================

print_header("VALIDASI PREDICTION CASE")

display(

    prediction_df[
        "prediction_case"
    ].value_counts()

)

print()

print_success(
    "Prediction case berhasil divalidasi."
)

VALIDASI PREDICTION CASE


,count
prediction_case,
Benar_Positif,1228
Benar_Negatif,554
Salah_Positif_ke_Negatif,55
Salah_Netral_ke_Negatif,51
Salah_Negatif_ke_Positif,30
Salah_Negatif_ke_Netral,29
Salah_Netral_ke_Positif,16
Salah_Positif_ke_Netral,9
Benar_Netral,8



✅ Prediction case berhasil divalidasi.


# 4. Analisis Distribusi Hasil Prediksi

## 4.1 Distribusi Prediction Case

In [ ]:
# =====================================================
# CELL 13 : DISTRIBUSI PREDICTION CASE
# =====================================================

print_header("DISTRIBUSI PREDICTION CASE")

case_distribution = (
    prediction_df["prediction_case"]
    .value_counts()
    .rename_axis("prediction_case")
    .reset_index(name="jumlah")
)

display(case_distribution)

print()

print_info(
    f"Total kategori : {len(case_distribution)}"
)

print_success(
    "Distribusi prediction case berhasil dibuat."
)

DISTRIBUSI PREDICTION CASE


,prediction_case,jumlah
0,Benar_Positif,1228
1,Benar_Negatif,554
2,Salah_Positif_ke_Negatif,55
3,Salah_Netral_ke_Negatif,51
4,Salah_Negatif_ke_Positif,30
5,Salah_Negatif_ke_Netral,29
6,Salah_Netral_ke_Positif,16
7,Salah_Positif_ke_Netral,9
8,Benar_Netral,8



ℹ️ Total kategori : 9
✅ Distribusi prediction case berhasil dibuat.


## 4.2 Distribusi Confidence Level

In [ ]:
# =====================================================
# CELL 14 : DISTRIBUSI CONFIDENCE LEVEL
# =====================================================

print_header("DISTRIBUSI CONFIDENCE LEVEL")

confidence_distribution = (
    prediction_df["confidence_level"]
    .value_counts()
    .rename_axis("confidence_level")
    .reset_index(name="jumlah")
)

display(confidence_distribution)

print_success(
    "Distribusi confidence level berhasil dibuat."
)

DISTRIBUSI CONFIDENCE LEVEL


,confidence_level,jumlah
0,Tinggi,1681
1,Sedang,153
2,Rendah,146


✅ Distribusi confidence level berhasil dibuat.


## 4.3 Statistik Confidence

In [ ]:
# =====================================================
# CELL 15 : STATISTIK CONFIDENCE
# =====================================================

print_header("STATISTIK CONFIDENCE")

confidence_summary = (
    prediction_df
    .groupby("prediction_case")["confidence"]
    .agg(
        jumlah="count",
        minimum="min",
        rata_rata="mean",
        maksimum="max",
        std="std"
    )
    .reset_index()
)

confidence_summary = confidence_summary.round(4)

display(confidence_summary)

print_success(
    "Statistik confidence berhasil dihitung."
)

STATISTIK CONFIDENCE


,prediction_case,jumlah,minimum,rata_rata,maksimum,std
0,Benar_Negatif,554,0.3854,0.9053,0.9919,0.1229
1,Benar_Netral,8,0.3885,0.5662,0.7848,0.1638
2,Benar_Positif,1228,0.3853,0.9902,0.9982,0.0528
3,Salah_Negatif_ke_Netral,29,0.3710,0.5138,0.6547,0.0775
4,Salah_Negatif_ke_Positif,30,0.4083,0.7975,0.9981,0.1842
5,Salah_Netral_ke_Negatif,51,0.4735,0.8579,0.9841,0.1340
6,Salah_Netral_ke_Positif,16,0.5031,0.8615,0.9981,0.1712
7,Salah_Positif_ke_Negatif,55,0.3623,0.7536,0.9839,0.2089
8,Salah_Positif_ke_Netral,9,0.3727,0.5531,0.7959,0.1281


✅ Statistik confidence berhasil dihitung.


## 4.4 Menyimpan Hasil Distribusi

In [ ]:
# =====================================================
# CELL 16 : SIMPAN DISTRIBUSI
# =====================================================

print_header("MENYIMPAN DISTRIBUSI")

case_distribution.to_csv(
    os.path.join(
        TABLE_DIR,
        "distribution_prediction_case.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

confidence_distribution.to_csv(
    os.path.join(
        TABLE_DIR,
        "distribution_confidence_level.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

confidence_summary.to_csv(
    os.path.join(
        TABLE_DIR,
        "confidence_summary.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print_success(
    "Seluruh tabel distribusi berhasil disimpan."
)

MENYIMPAN DISTRIBUSI
✅ Seluruh tabel distribusi berhasil disimpan.


# 5. Purposive Sampling

## 5.1 Menentukan Target Sampling

In [ ]:
# =====================================================
# CELL 17 : TARGET PURPOSIVE SAMPLING
# =====================================================

print_header("TARGET PURPOSIVE SAMPLING")

sampling_targets = {

    "Benar_Positif": 5,
    "Benar_Negatif": 5,
    "Benar_Netral": 5,

    "Salah_Netral_ke_Positif": 5,
    "Salah_Netral_ke_Negatif": 5

}

target_df = pd.DataFrame(

    sampling_targets.items(),

    columns=[
        "prediction_case",
        "target_sample"
    ]

)

display(target_df)

print_success(
    "Target sampling berhasil dibuat."
)

TARGET PURPOSIVE SAMPLING


,prediction_case,target_sample
0,Benar_Positif,5
1,Benar_Negatif,5
2,Benar_Netral,5
3,Salah_Netral_ke_Positif,5
4,Salah_Netral_ke_Negatif,5


✅ Target sampling berhasil dibuat.


## 5.2 Fungsi Purposive Sampling

In [ ]:
# =====================================================
# CELL 18 : FUNGSI PURPOSIVE SAMPLING
# =====================================================

print_header("MEMBUAT FUNGSI PURPOSIVE SAMPLING")


def purposive_sampling(

    dataframe,
    prediction_case,
    n_samples=5

):

    """
    Memilih sampel berdasarkan prediction_case.

    Prioritas:
    1. Confidence tertinggi
    2. Tidak ada text yang duplikat
    """

    temp = (

        dataframe[
            dataframe["prediction_case"] == prediction_case
        ]

        .sort_values(

            by="confidence",

            ascending=False

        )

        .drop_duplicates(

            subset="text"

        )

    )

    available = len(temp)

    selected = min(

        n_samples,

        available

    )

    sample = temp.head(selected)

    return sample, available


print_success(
    "Fungsi purposive sampling berhasil dibuat."
)

MEMBUAT FUNGSI PURPOSIVE SAMPLING
✅ Fungsi purposive sampling berhasil dibuat.


## 5.3 Melakukan Purposive Sampling

In [ ]:
# =====================================================
# CELL 19 : MEMILIH SAMPEL ANALISIS
# =====================================================

print_header("MEMILIH SAMPEL ANALISIS")

sampling_plan = {

    "Benar_Positif": 5,

    "Benar_Negatif": 5,

    "Benar_Netral": 5,

    "Salah_Netral_ke_Positif": 5,

    "Salah_Netral_ke_Negatif": 5

}

selected_reviews = []

sampling_summary = []

used_text = set()

for prediction_case, target in sampling_plan.items():

    sample, available = purposive_sampling(

        prediction_df,

        prediction_case,

        target * 3

    )

    final_rows = []

    for _, row in sample.iterrows():

        if row["text"] in used_text:

            continue

        final_rows.append(row)

        used_text.add(row["text"])

        if len(final_rows) == target:

            break

    final_df = pd.DataFrame(final_rows)

    selected_reviews.append(final_df)

    sampling_summary.append(

        {

            "prediction_case": prediction_case,

            "target": target,

            "available": available,

            "selected": len(final_df)

        }

    )

selected_reviews_df = (
    pd.concat(
        selected_reviews,
        ignore_index=True
    )
)

sampling_log_df = pd.DataFrame(
    sampling_summary
)

print_success(
    "Purposive sampling selesai."
)

display(
    sampling_log_df
)

MEMILIH SAMPEL ANALISIS
✅ Purposive sampling selesai.


,prediction_case,target,available,selected
0,Benar_Positif,5,752,5
1,Benar_Negatif,5,551,5
2,Benar_Netral,5,8,5
3,Salah_Netral_ke_Positif,5,16,5
4,Salah_Netral_ke_Negatif,5,51,5


## 5.4 Validasi Hasil Sampling

In [ ]:
# =====================================================
# CELL 20 : VALIDASI SAMPLING
# =====================================================

print_header("VALIDASI SAMPLING")

display(
    sampling_log_df
)

print()

print_info(
    f"Jumlah Sampel : {len(selected_reviews_df)}"
)

print()

display(

    selected_reviews_df[
        "prediction_case"
    ].value_counts()

)

# ==========================
# Validasi Text Duplikat
# ==========================

duplicate_text = (
    selected_reviews_df["text"]
    .duplicated()
    .sum()
)

print()

print_info(
    f"Jumlah Text Duplikat : {duplicate_text}"
)

if duplicate_text == 0:

    print_success(
        "Tidak ditemukan text duplikat."
    )

else:

    print_warning(
        f"Ditemukan {duplicate_text} text duplikat."
    )

print_success(
    "Validasi selesai."
)

VALIDASI SAMPLING


,prediction_case,target,available,selected
0,Benar_Positif,5,752,5
1,Benar_Negatif,5,551,5
2,Benar_Netral,5,8,5
3,Salah_Netral_ke_Positif,5,16,5
4,Salah_Netral_ke_Negatif,5,51,5



ℹ️ Jumlah Sampel : 25



,count
prediction_case,
Benar_Positif,5
Benar_Negatif,5
Benar_Netral,5
Salah_Netral_ke_Positif,5
Salah_Netral_ke_Negatif,5



ℹ️ Jumlah Text Duplikat : 0
✅ Tidak ditemukan text duplikat.
✅ Validasi selesai.


# 6. Penyusunan Dataset Final XAI

## 6.1 Memilih Sampel Confidence Rendah


In [ ]:
# =====================================================
# CELL 21 : MEMILIH SAMPEL CONFIDENCE RENDAH
# =====================================================

print_header("MEMILIH SAMPEL CONFIDENCE RENDAH")

# Semua prediksi yang BENAR
correct_df = prediction_df[
    prediction_df["actual_sentiment"] ==
    prediction_df["predicted_sentiment"]
].copy()

# Urutkan dari confidence paling rendah
correct_df = correct_df.sort_values(
    by="confidence",
    ascending=True
)

# Text yang SUDAH dipilih pada purposive sampling
selected_text = set(
    selected_reviews_df["text"].astype(str)
)

# Menyimpan sampel baru
low_confidence_rows = []

for _, row in correct_df.iterrows():

    if row["text"] in selected_text:
        continue

    low_confidence_rows.append(row)

    selected_text.add(row["text"])

    if len(low_confidence_rows) == 5:
        break

low_confidence_df = pd.DataFrame(
    low_confidence_rows
)

low_confidence_df["prediction_case"] = (
    "Benar_Confidence_Rendah"
)

print_success(
    f"Jumlah sampel confidence rendah : {len(low_confidence_df)}"
)

display(
    low_confidence_df[
        [
            "actual_sentiment",
            "predicted_sentiment",
            "confidence"
        ]
    ]
)

MEMILIH SAMPEL CONFIDENCE RENDAH
✅ Jumlah sampel confidence rendah : 5


,actual_sentiment,predicted_sentiment,confidence
1284,Positif,Positif,0.385265
840,Negatif,Negatif,0.385356
1033,Netral,Netral,0.388543
197,Netral,Netral,0.393925
313,Negatif,Negatif,0.402289


## 6.2 Menggabungkan Dataset Final

In [ ]:
# =====================================================
# CELL 22 : MENGGABUNGKAN DATASET FINAL
# =====================================================

print_header("MENGGABUNGKAN DATASET FINAL")

selected_reviews_df = pd.concat(

    [
        selected_reviews_df,
        low_confidence_df
    ],

    ignore_index=True

)

selected_reviews_df = selected_reviews_df.reset_index(
    drop=True
)

print_info(
    f"Jumlah sampel akhir : {len(selected_reviews_df)}"
)

print_success(
    "Dataset final berhasil dibuat."
)

MENGGABUNGKAN DATASET FINAL
ℹ️ Jumlah sampel akhir : 30
✅ Dataset final berhasil dibuat.


## 6.3 Membuat Identitas Sampel

In [ ]:
# =====================================================
# CELL 23 : MEMBUAT IDENTITAS SAMPEL
# =====================================================

print_header("MEMBUAT IDENTITAS SAMPEL")

# Hapus jika sudah ada
for col in ["analysis_order", "sample_id"]:

    if col in selected_reviews_df.columns:

        selected_reviews_df.drop(
            columns=col,
            inplace=True
        )

selected_reviews_df.insert(

    0,

    "analysis_order",

    range(
        1,
        len(selected_reviews_df)+1
    )

)

selected_reviews_df.insert(

    1,

    "sample_id",

    [
        f"XAI_{i:03d}"

        for i in range(
            1,
            len(selected_reviews_df)+1
        )
    ]

)

print_success(
    "Sample ID berhasil dibuat."
)

display(
    selected_reviews_df.head()
)

MEMBUAT IDENTITAS SAMPEL
✅ Sample ID berhasil dibuat.


,analysis_order,sample_id,text,actual_label,actual_sentiment,predicted_label,predicted_sentiment,prob_negatif,prob_netral,prob_positif,confidence,correct,prediction_case,confidence_level
0,1,XAI_001,sangat membantu dan mudah,2,Positif,2,Positif,0.000885,0.000943,0.998172,0.998172,True,Benar_Positif,Tinggi
1,2,XAI_002,mudah dan sangat membantu sekali,2,Positif,2,Positif,0.000879,0.000953,0.998167,0.998167,True,Benar_Positif,Tinggi
2,3,XAI_003,sangat membantu sekali,2,Positif,2,Positif,0.000898,0.000940,0.998162,0.998162,True,Benar_Positif,Tinggi
3,4,XAI_004,baik dan sangat membantu,2,Positif,2,Positif,0.000917,0.000924,0.998159,0.998159,True,Benar_Positif,Tinggi
4,5,XAI_005,sangat membantu sekali dalam segala aktivitas,2,Positif,2,Positif,0.000891,0.000954,0.998155,0.998155,True,Benar_Positif,Tinggi


## 6.4 Menambahkan Metadata Analisis

In [ ]:
# =====================================================
# CELL 24 : MENAMBAHKAN METADATA
# =====================================================

print_header("MENAMBAHKAN METADATA")

reason_mapping = {

    "Benar_Positif":
        "Prediksi benar kelas Positif",

    "Benar_Negatif":
        "Prediksi benar kelas Negatif",

    "Benar_Netral":
        "Prediksi benar kelas Netral",

    "Salah_Netral_ke_Positif":
        "Prediksi salah: Netral menjadi Positif",

    "Salah_Netral_ke_Negatif":
        "Prediksi salah: Netral menjadi Negatif",

    "Benar_Confidence_Rendah":
        "Prediksi benar dengan confidence rendah"

}

selected_reviews_df["selection_reason"] = (
    selected_reviews_df["prediction_case"]
    .map(reason_mapping)
)

def get_sample_group(case):

    if case == "Benar_Confidence_Rendah":
        return "Low Confidence"

    elif case.startswith("Salah"):
        return "Error Analysis"

    return "High Confidence"

selected_reviews_df["sample_group"] = (
    selected_reviews_df["prediction_case"]
    .apply(get_sample_group)
)

selected_reviews_df["explain_status"] = "Pending"

print_success(
    "Metadata berhasil dibuat."
)

MENAMBAHKAN METADATA
✅ Metadata berhasil dibuat.


## 6.5 Validasi Dataset Final

In [ ]:
# =====================================================
# CELL 25 : VALIDASI DATASET FINAL
# =====================================================

print_header("VALIDASI DATASET FINAL")

duplicate_text = (
    selected_reviews_df["text"]
    .duplicated()
    .sum()
)

print_info(
    f"Jumlah Sampel : {len(selected_reviews_df)}"
)

print_info(
    f"Jumlah Duplikat : {duplicate_text}"
)

print()

display(
    selected_reviews_df[
        "sample_group"
    ].value_counts()
)

print()

display(
    selected_reviews_df[
        "prediction_case"
    ].value_counts()
)

if len(selected_reviews_df) == 30 and duplicate_text == 0:

    print_success(
        "Dataset final siap digunakan untuk SHAP dan LIME."
    )

else:

    print_warning(
        "Periksa kembali hasil dataset."
    )

VALIDASI DATASET FINAL
ℹ️ Jumlah Sampel : 30
ℹ️ Jumlah Duplikat : 0



,count
sample_group,
High Confidence,15
Error Analysis,10
Low Confidence,5


,count
prediction_case,
Benar_Positif,5
Benar_Negatif,5
Benar_Netral,5
Salah_Netral_ke_Positif,5
Salah_Netral_ke_Negatif,5
Benar_Confidence_Rendah,5


✅ Dataset final siap digunakan untuk SHAP dan LIME.


# 7. Menyimpan Hasil Purposive Sampling

## 7.1 Menyiapkan Folder Penyimpanan

In [ ]:
# =====================================================
# CELL 26 : MENYIAPKAN FOLDER PENYIMPANAN
# =====================================================

print_header("MENYIAPKAN FOLDER PENYIMPANAN")

OUTPUT_DIR = os.path.join(

    PROJECT_DIR,

    "xai",

    "XB_Persiapan_XAI"

)

CSV_DIR = os.path.join(

    OUTPUT_DIR,

    "csv"

)

LOG_DIR = os.path.join(

    OUTPUT_DIR,

    "log"

)

DOC_DIR = os.path.join(

    OUTPUT_DIR,

    "documentation"

)

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(DOC_DIR, exist_ok=True)

print_success("Folder berhasil dibuat.")

print()

print_info(f"Output Folder : {OUTPUT_DIR}")
print_info(f"CSV Folder    : {CSV_DIR}")
print_info(f"LOG Folder    : {LOG_DIR}")
print_info(f"DOC Folder    : {DOC_DIR}")

MENYIAPKAN FOLDER PENYIMPANAN
✅ Folder berhasil dibuat.

ℹ️ Output Folder : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI
ℹ️ CSV Folder    : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv
ℹ️ LOG Folder    : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/log
ℹ️ DOC Folder    : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/documentation


## 7.2 Menyimpan Dataset Final XAI

In [ ]:
# =====================================================
# CELL 27 : MENYIMPAN DATASET FINAL
# =====================================================

print_header("MENYIMPAN DATASET FINAL")

SELECTED_REVIEWS_FILE = os.path.join(

    CSV_DIR,

    "selected_reviews_xai.csv"

)

selected_reviews_df.to_csv(

    SELECTED_REVIEWS_FILE,

    index=False,

    encoding="utf-8-sig"

)

print_success(
    "Dataset final berhasil disimpan."
)

print()

print_info(
    SELECTED_REVIEWS_FILE
)

MENYIMPAN DATASET FINAL
✅ Dataset final berhasil disimpan.

ℹ️ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv/selected_reviews_xai.csv


## 7.3 Menyimpan Log Purposive Sampling

In [ ]:
# =====================================================
# CELL 28 : MENYIMPAN LOG SAMPLING
# =====================================================

print_header("MENYIMPAN LOG SAMPLING")

SAMPLING_LOG_FILE = os.path.join(

    LOG_DIR,

    "sampling_log.csv"

)

sampling_log_df.to_csv(

    SAMPLING_LOG_FILE,

    index=False,

    encoding="utf-8-sig"

)

print_success(
    "Sampling log berhasil disimpan."
)

print()

print_info(
    SAMPLING_LOG_FILE
)

MENYIMPAN LOG SAMPLING
✅ Sampling log berhasil disimpan.

ℹ️ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/log/sampling_log.csv


## 7.4 Menyimpan Distribusi Sampel

In [ ]:
# =====================================================
# CELL 29 : MENYIMPAN DISTRIBUSI SAMPEL
# =====================================================

print_header("MENYIMPAN DISTRIBUSI SAMPEL")

distribution_df = (

    selected_reviews_df

    .groupby(

        [

            "sample_group",

            "prediction_case"

        ]

    )

    .size()

    .reset_index(

        name="count"

    )

)

DISTRIBUTION_FILE = os.path.join(

    LOG_DIR,

    "sampling_distribution.csv"

)

distribution_df.to_csv(

    DISTRIBUTION_FILE,

    index=False,

    encoding="utf-8-sig"

)

print_success(
    "Distribusi sampel berhasil disimpan."
)

display(distribution_df)

MENYIMPAN DISTRIBUSI SAMPEL
✅ Distribusi sampel berhasil disimpan.


,sample_group,prediction_case,count
0,Error Analysis,Salah_Netral_ke_Negatif,5
1,Error Analysis,Salah_Netral_ke_Positif,5
2,High Confidence,Benar_Negatif,5
3,High Confidence,Benar_Netral,5
4,High Confidence,Benar_Positif,5
5,Low Confidence,Benar_Confidence_Rendah,5


## 7.5 Membuat Dokumentasi Notebook

In [ ]:
# =====================================================
# CELL 30 : MEMBUAT DOKUMENTASI
# =====================================================

print_header("MEMBUAT DOKUMENTASI")

readme_text = """
==================================================
NOTEBOOK XB
Persiapan Dataset Final XAI
==================================================

Notebook :
XB_Skripsi_IndoBERT_Purposive_Sampling_XAI.ipynb

Output yang dihasilkan

1. selected_reviews_xai.csv
   Dataset final berisi 30 ulasan untuk analisis SHAP dan LIME.

2. sampling_log.csv
   Ringkasan proses purposive sampling.

3. sampling_distribution.csv
   Distribusi sampel berdasarkan kategori analisis.

Notebook selanjutnya

XC_Skripsi_IndoBERT_SHAP.ipynb

==================================================
"""

README_FILE = os.path.join(

    DOC_DIR,

    "README.txt"

)

with open(

    README_FILE,

    "w",

    encoding="utf-8"

) as f:

    f.write(readme_text)

print_success(
    "README berhasil dibuat."
)

print()

print_info(
    README_FILE
)

MEMBUAT DOKUMENTASI
✅ README berhasil dibuat.

ℹ️ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/documentation/README.txt


## 7.6 Validasi Penyimpanan

In [ ]:
# =====================================================
# CELL 31 : VALIDASI PENYIMPANAN
# =====================================================

print_header("VALIDASI PENYIMPANAN")

files_to_check = [

    SELECTED_REVIEWS_FILE,

    SAMPLING_LOG_FILE,

    DISTRIBUTION_FILE,

    README_FILE

]

for file in files_to_check:

    if os.path.exists(file):

        print_success(

            f"Tersimpan : {os.path.basename(file)}"

        )

    else:

        print_warning(

            f"Tidak ditemukan : {os.path.basename(file)}"

        )

print()

print_success(
    "Seluruh output Notebook XB berhasil disimpan."
)

VALIDASI PENYIMPANAN
✅ Tersimpan : selected_reviews_xai.csv
✅ Tersimpan : sampling_log.csv
✅ Tersimpan : sampling_distribution.csv
✅ Tersimpan : README.txt

✅ Seluruh output Notebook XB berhasil disimpan.


# 8. Ringkasan Notebook

## 8.1 Ringkasan Hasil Notebook

In [ ]:
# =====================================================
# CELL 32 : RINGKASAN NOTEBOOK
# =====================================================

print_header("RINGKASAN NOTEBOOK")

print("Notebook                : XB")
print("Status                  : Selesai")
print()

print("Output yang dihasilkan")
print("- test_predictions_xai.csv (dari Notebook XA)")
print("- selected_reviews_xai.csv")
print("- sampling_log.csv")
print("- sampling_distribution.csv")
print("- README.txt")

print()

print("Notebook selanjutnya")
print("- XC_Skripsi_IndoBERT_SHAP_Analysis.ipynb")

print()

print_success(
    "Notebook XB selesai dan siap digunakan."
)

RINGKASAN NOTEBOOK
Notebook                : XB
Status                  : Selesai

Output yang dihasilkan
- test_predictions_xai.csv (dari Notebook XA)
- selected_reviews_xai.csv
- sampling_log.csv
- sampling_distribution.csv
- README.txt

Notebook selanjutnya
- XC_Skripsi_IndoBERT_SHAP_Analysis.ipynb

✅ Notebook XB selesai dan siap digunakan.
